# Build the ColorCount APK (free, in the browser)

Google Colab gives you a free Linux computer, which is what the Android build tool (Buildozer) needs.

**Run the cells from top to bottom** (click each ▶ button and wait for it to finish):

1. **Install the build tools.** Takes about 5 minutes.
2. **Upload `ColorCountAndroid.zip`.**
3. **Build the APK.** The first build takes about **40–60 minutes**. Keep this tab open.
4. **Download the APK.**

Before you start, set *Runtime → Change runtime type* to **CPU**. A GPU isn't needed.

In [ ]:
# 1) Install the build tools  (~5 min) - for Colab's Ubuntu 24.04 / Python 3.13
!sudo apt-get update -qq
!sudo DEBIAN_FRONTEND=noninteractive apt-get install -y -qq git zip unzip openjdk-17-jdk autoconf automake libtool pkg-config zlib1g-dev libncurses-dev libffi-dev libssl-dev cmake build-essential ccache libltdl-dev > /tmp/apt.log 2>&1 && echo "✅ System packages installed" || (echo "❌ apt failed:"; tail -n 20 /tmp/apt.log)
# older Android tools look for these older library names
!for L in libtinfo libncurses; do [ -e /usr/lib/x86_64-linux-gnu/$L.so.5 ] || sudo ln -sf /usr/lib/x86_64-linux-gnu/$L.so.6 /usr/lib/x86_64-linux-gnu/$L.so.5; done

# Buildozer needs Python 3.11 (Colab has 3.13), so it gets its own environment
!pip install -q uv
!uv venv -q --seed -p 3.11 /content/bvenv
!uv pip install -q -p /content/bvenv/bin/python buildozer "cython==0.29.37" setuptools

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["VIRTUAL_ENV"] = "/content/bvenv"
os.environ["PATH"] = "/content/bvenv/bin:" + os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]
!java -version 2>&1 | head -n 1
!/content/bvenv/bin/python --version
!buildozer --version
print("\n✅ Tools installed")

In [ ]:
# 2) Upload ColorCountAndroid.zip  (a "Choose Files" button appears below)
from google.colab import files
import os, glob, zipfile, shutil

os.chdir("/content")
uploaded = files.upload()
name = next(iter(uploaded))
# extract over the old copy (keeps the compiled .buildozer folder, so rebuilds are fast)
zipfile.ZipFile(name).extractall("/content/app")
spec = glob.glob("/content/app/**/buildozer.spec", recursive=True)
assert spec, "buildozer.spec not found in the zip - upload ColorCountAndroid.zip"
APP_DIR = os.path.dirname(spec[0])
os.chdir(APP_DIR)
print("✅ Project ready in", APP_DIR)
!ls

In [ ]:
# 3) Build the APK  (first time ~40-60 min; keep this tab open)
import os
os.chdir(APP_DIR)
!yes | buildozer -v android debug 2>&1 | tee build.log | grep --line-buffered -E "^\[INFO\]|^# |BUILD SUCCESSFUL|Error|error:" 
!ls -la bin/ 2>/dev/null || echo "❌ No APK yet - run the next cell to see what went wrong"

In [ ]:
# 4) Download the APK to your computer
import glob
from google.colab import files
apks = glob.glob(APP_DIR + "/bin/*.apk")
if apks:
    print("✅ Built:", apks[0])
    files.download(apks[0])
else:
    print("❌ The build failed. The last 80 lines of the log are below; send them to Claude:\n")
    !tail -n 80 {APP_DIR}/build.log

### Rebuilding after a code change
Upload the new zip again (cell 2) and rerun cells 3 and 4. A rebuild in the same session only takes a few minutes, because the Android tools and OpenCV are already compiled.